In [393]:
# eita yatim cell ei cell er sathe uporer cell er kono relation nai.
query = "south korea"
choices = [
    "south korea",
    "korea south",
    "north korea",
    "south africa"
]

process.extract(
    query,
    choices,
    limit=4,
    scorer=fuzz.token_sort_ratio
)

[('south korea', 100),
 ('korea south', 100),
 ('north korea', 82),
 ('south africa', 70)]

In [394]:
import numpy as np
import pandas as pd

from thefuzz import process, fuzz

np.random.seed(42)

In [395]:
canonical_cities = [
    'dhaka',
    'chittagong',
    'sylhet',
    'rajshahi',
    'khulna',
    'barisal',
    'rangpur',
    'comilla',
    'mymensingh',
    'coxsbazar'
]

# Realistic inconsistent entries
city_variations = [
    'Dhaka', 'DHAKA', ' dhaka', 'dhaka ',
    'Chittagong', 'CHITTAGONG', ' chittagong', 'chitagonj', 'chatgram',
    'Sylhet', 'SYLHET', ' sylhet', 'sylhett',
    'Rajshahi', 'RAJSHAHI', ' rajshahi', 'rajshai',
    'Khulna', 'KHULNA', ' khulna', 'khulnaa',
    'Barisal', 'BARISAL', ' barisal', 'borishal',
    'Rangpur', 'RANGPUR', ' rangpur', 'rangpur ',
    'Comilla', 'COMILLA', ' comilla', 'cumilla',
    'Mymensingh', 'MYMENSINGH', ' mymensingh', 'mymensing',
    "Cox's Bazar", "COX'S BAZAR", " cox's bazar", 'coxsbazar'
]

n_rows = 2000

df = pd.DataFrame({
    'customer_id': np.arange(10001, 10001 + n_rows),
    'city': np.random.choice(city_variations, size=n_rows),
    'loan_amount': np.random.randint(10000, 500000, size=n_rows),
    'age': np.random.randint(20, 65, size=n_rows)
})

df.head()

,customer_id,city,loan_amount,age
0,10001,COX'S BAZAR,206553,43
1,10002,rangpur,471792,63
2,10003,RAJSHAHI,270337,30
3,10004,chitagonj,137556,36
4,10005,khulnaa,205246,23


## investigate the dataset

In [396]:
# amar kache 2k ta row and 4 ta column ache.
df.shape

(2000, 4)

In [397]:
df.head()

,customer_id,city,loan_amount,age
0,10001,COX'S BAZAR,206553,43
1,10002,rangpur,471792,63
2,10003,RAJSHAHI,270337,30
3,10004,chitagonj,137556,36
4,10005,khulnaa,205246,23


In [398]:
# amra shudhu matro city column ta niye kaj korbo karon etar data type string
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  2000 non-null   int64
 1   city         2000 non-null   str  
 2   loan_amount  2000 non-null   int64
 3   age          2000 non-null   int64
dtypes: int64(3), str(1)
memory usage: 62.6 KB


## identify inconsistent values

In [399]:
# check korbo je kono inconsistent data ache kina

np.sort(df['city'].unique())

array([' barisal', ' chittagong', ' comilla', " cox's bazar", ' dhaka',
       ' khulna', ' mymensingh', ' rajshahi', ' rangpur', ' sylhet',
       'BARISAL', 'Barisal', 'CHITTAGONG', 'COMILLA', "COX'S BAZAR",
       'Chittagong', 'Comilla', "Cox's Bazar", 'DHAKA', 'Dhaka', 'KHULNA',
       'Khulna', 'MYMENSINGH', 'Mymensingh', 'RAJSHAHI', 'RANGPUR',
       'Rajshahi', 'Rangpur', 'SYLHET', 'Sylhet', 'borishal', 'chatgram',
       'chitagonj', 'coxsbazar', 'cumilla', 'dhaka ', 'khulnaa',
       'mymensing', 'rajshai', 'rangpur ', 'sylhett'], dtype=object)

In [400]:
df['city'].nunique()

41

In [401]:
# uporer cell theke dekhlam je inconsistent data ase and total 41 ta unique values ase. ekhon just value count check korbo.
# note: ei cell e inconsistent data + value + value count shobi dekha jay.but uporer ta ekta array hishebe ditese tai uporer cell er output dekhte shubidha hoy.

df['city'].value_counts().sort_index()

city
 barisal        52
 chittagong     39
 comilla        53
 cox's bazar    48
 dhaka          47
 khulna         46
 mymensingh     51
 rajshahi       39
 rangpur        53
 sylhet         53
BARISAL         51
Barisal         47
CHITTAGONG      49
COMILLA         37
COX'S BAZAR     50
Chittagong      49
Comilla         48
Cox's Bazar     41
DHAKA           51
Dhaka           50
KHULNA          41
Khulna          44
MYMENSINGH      56
Mymensingh      51
RAJSHAHI        50
RANGPUR         34
Rajshahi        47
Rangpur         67
SYLHET          49
Sylhet          44
borishal        60
chatgram        45
chitagonj       56
coxsbazar       43
cumilla         59
dhaka           44
khulnaa         42
mymensing       60
rajshai         50
rangpur         59
sylhett         45
Name: count, dtype: int64

## basic text normalization

In [402]:
# amra dataset niye enough idea niye niyechi. ekhon basic data cleaning shuru korbo.
# first target holo shobgula data ke lowercase e nibo. then whitespace remove korbo. and eigula main dataset er city column e sathe sathe update kore dibo. erpor city column er unique values abar check korbo. dekhbo je inconsitency onektukui solve hoye geche.

pass

In [403]:
# Convert all text to lowercase
df['city'] = df['city'].str.lower()

# whitespace remove er jonne strip() method use kora hoy
df['city'] = df['city'].str.strip()

In [404]:
# ekhon city column er unique value abar check kortesi.

df['city'].nunique()

# joss aage unique value chilo 42 ta ekhon 19 ta

19

In [405]:
# ami upper case, lower case, whitespace er problem solve korchi. but ekhon jeigula ase oigual spelling mistake. eigula solve korte hole amake fuzzy matching er help nite hobe.
np.sort(df['city'].unique())

array(['barisal', 'borishal', 'chatgram', 'chitagonj', 'chittagong',
       'comilla', "cox's bazar", 'coxsbazar', 'cumilla', 'dhaka',
       'khulna', 'khulnaa', 'mymensing', 'mymensingh', 'rajshahi',
       'rajshai', 'rangpur', 'sylhet', 'sylhett'], dtype=object)

## Fuzzy Matching

In [406]:
# fuzzy matching use kortesi jeigula banan vul korse oigula correct korar jonne.
# amra protita city eri spelling thik korbo fuzzy matching diye. but shurute just ekta sample(chittagong) niye kaj kore dekhbo je result kemon ashe. etay main dataset change korbo na. just process ta dekhbo fuzzy er.

### just an example with a single sample to understand the process

In [407]:
target_city = "chittagong"

matches = process.extract(
    target_city,
    df['city'].unique(),
    limit=5,
    scorer=fuzz.ratio
)
# matches er moddhe ami ekta list of tuples pabo. matches[0] = ('chittagong', 100) and matches[0][0] = 'chittagong' and matches[0][1] = 100.
matches

[('chittagong', 100),
 ('chitagonj', 84),
 ('chatgram', 44),
 ('rangpur', 35),
 ('comilla', 35)]

In [408]:
# uporer score gula dekhe amake decide korte hobe threshold value ami koto set korbo. industry standard holo threshold value 70 to 80 er moddhe rakha.
# then threshold set kore new ekta empty array te threshold pass kora city gulare store korbo.

threshold = 70
valid_matches = []

for match in matches:
    word = match[0]
    score = match[1]

    if score >= threshold:
        valid_matches.append((word, score))

# valid_matches er moddhe threshold pass kora city gula ase. and yes amra chittagong er spelling mistake gulare correct korte parsi except 'chatgram' 
valid_matches

[('chittagong', 100), ('chitagonj', 84)]

### real implementation of fuzzy matching

In [409]:
# better approch

unique_cities = df['city'].unique()

city_matches = []

for city in unique_cities:
    match = process.extractOne(
        city,
        canonical_cities,
        scorer=fuzz.ratio
    )

    city_matches.append({
        'original': city,
        'matched': match[0],
        'score': match[1]
    })

matches_df = pd.DataFrame(city_matches)

matches_df.sort_values(
    by='score',
    ascending=True
)

,original,matched,score
15,chatgram,dhaka,46
12,borishal,barisal,80
3,chitagonj,chittagong,84
11,cumilla,comilla,86
0,cox's bazar,coxsbazar,90
4,khulnaa,khulna,92
18,sylhett,sylhet,92
16,rajshai,rajshahi,93
13,mymensing,mymensingh,95
6,barisal,barisal,100


### apply threshold in all matches

In [410]:
threshold = 70

matches_df['accepted'] = matches_df['score'] >= threshold

matches_df.sort_values(
    by='score',
    ascending=True
)

,original,matched,score,accepted
15,chatgram,dhaka,46,False
12,borishal,barisal,80,True
3,chitagonj,chittagong,84,True
11,cumilla,comilla,86,True
0,cox's bazar,coxsbazar,90,True
4,khulnaa,khulna,92,True
18,sylhett,sylhet,92,True
16,rajshai,rajshahi,93,True
13,mymensing,mymensingh,95,True
6,barisal,barisal,100,True


### only accepted matches

In [411]:
accepted_matches = matches_df[
    matches_df['accepted']
].copy()

accepted_matches.sort_values(
    by='score',
    ascending=True
)

,original,matched,score,accepted
12,borishal,barisal,80,True
3,chitagonj,chittagong,84,True
11,cumilla,comilla,86,True
0,cox's bazar,coxsbazar,90,True
4,khulnaa,khulna,92,True
18,sylhett,sylhet,92,True
16,rajshai,rajshahi,93,True
13,mymensing,mymensingh,95,True
7,sylhet,sylhet,100,True
6,barisal,barisal,100,True


In [412]:
review_required = matches_df[
    matches_df['score'] < threshold
].copy()

review_required.sort_values(
    by='score',
    ascending=True
)

,original,matched,score,accepted
15,chatgram,dhaka,46,False


In [413]:
mapping = dict(
    zip(
        accepted_matches['original'],
        accepted_matches['matched']
    )
)

mapping

{"cox's bazar": 'coxsbazar',
 'rangpur': 'rangpur',
 'rajshahi': 'rajshahi',
 'chitagonj': 'chittagong',
 'khulnaa': 'khulna',
 'khulna': 'khulna',
 'barisal': 'barisal',
 'sylhet': 'sylhet',
 'mymensingh': 'mymensingh',
 'dhaka': 'dhaka',
 'comilla': 'comilla',
 'cumilla': 'comilla',
 'borishal': 'barisal',
 'mymensing': 'mymensingh',
 'chittagong': 'chittagong',
 'rajshai': 'rajshahi',
 'coxsbazar': 'coxsbazar',
 'sylhett': 'sylhet'}

In [414]:
mapping_df = pd.DataFrame(
    mapping.items(),
    columns=['original', 'standard']
)

mapping_df

,original,standard
0,cox's bazar,coxsbazar
1,rangpur,rangpur
2,rajshahi,rajshahi
3,chitagonj,chittagong
4,khulnaa,khulna
5,khulna,khulna
6,barisal,barisal
7,sylhet,sylhet
8,mymensingh,mymensingh
9,dhaka,dhaka


In [415]:
df['city'] = df['city'].replace(mapping)

In [416]:
np.sort(df['city'].unique())

array(['barisal', 'chatgram', 'chittagong', 'comilla', 'coxsbazar',
       'dhaka', 'khulna', 'mymensingh', 'rajshahi', 'rangpur', 'sylhet'],
      dtype=object)

In [417]:
df['city'].nunique()

11

In [418]:
df['city'].value_counts().sort_index()

city
barisal       210
chatgram       45
chittagong    193
comilla       197
coxsbazar     182
dhaka         192
khulna        173
mymensingh    218
rajshahi      186
rangpur       213
sylhet        191
Name: count, dtype: int64

In [419]:
# Final dataset-এর কোনো city canonical list-এর বাইরে আছে কি না
invalid_cities = sorted(
    set(df['city'].unique()) - set(canonical_cities)
)

invalid_cities

['chatgram']

In [420]:
df.head(10)

,customer_id,city,loan_amount,age
0,10001,coxsbazar,206553,43
1,10002,rangpur,471792,63
2,10003,rajshahi,270337,30
3,10004,chittagong,137556,36
4,10005,khulna,205246,23
5,10006,coxsbazar,36160,44
6,10007,khulna,256242,49
7,10008,barisal,27633,28
8,10009,sylhet,498096,40
9,10010,sylhet,289158,39
